# 🧠 Final Project 2 (v1) - MBDA
## ANN for EMNIST Alphabet Classification (A–Z)

**Dataset:** EMNIST Letters (26 classes: A–Z)
**Model:** Multilayer Perceptron (ANN)
**Acceleration:** GPU (NVIDIA CUDA)
**Visualization:** PyQt6 + Matplotlib Interactive

---

## 📦 1. Import Libraries

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import numpy as np
import pandas as pd
import time
import os

from sklearn.metrics import (
    confusion_matrix, classification_report, 
    accuracy_score, precision_recall_fscore_support
)

import matplotlib
matplotlib.use('QtAgg')  # PyQt6 backend for interactive display
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import seaborn as sns

print("✅ All libraries successfully imported!")
print(f"PyTorch version : {torch.__version__}")
print(f"Seaborn version : {sns.__version__}")

✅ All libraries successfully imported!
PyTorch version : 2.11.0+cu130
Seaborn version : 0.13.2


## ⚡ 2. GPU Detection and Configuration

In [2]:
# Automatic GPU detection
if torch.cuda.is_available():
    device = torch.device('cuda')
    gpu_name   = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU detected  : {gpu_name}")
    print(f"   VRAM Total      : {gpu_mem_gb:.2f} GB")
    print(f"   CUDA Version    : {torch.version.cuda}")
    print(f"   cuDNN Version   : {torch.backends.cudnn.version()}")
    # Enable benchmark mode for optimal convolution kernel selection
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device('cpu')
    print("⚠️ GPU not detected — using CPU.")

print(f"\n🖥️  Active device: {device}")

✅ GPU detected  : NVIDIA GeForce RTX 4050 Laptop GPU
   VRAM Total      : 6.44 GB
   CUDA Version    : 13.0
   cuDNN Version   : 91900

🖥️  Active device: cuda


## 📂 3. Load and Preprocessing EMNIST Letters Dataset

In [3]:
# EMNIST Letters: 26 classes (A–Z), 28x28 grayscale images
# Original EMNIST labels start from 1 (A=1 … Z=26)
# shift labels to be 0-indexed (A=0 … Z=25)

# Normalization: mean=0.1722, std=0.3309 (specific values for EMNIST Letters)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1722,), (0.3309,))
])

DATA_DIR = './data'

train_data = datasets.EMNIST(
    root=DATA_DIR, split='letters',
    train=True,  download=True, transform=transform
)
test_data  = datasets.EMNIST(
    root=DATA_DIR, split='letters',
    train=False, download=True, transform=transform
)

# Shift labels from 1–26 → 0–25
train_data.targets -= 1
test_data.targets  -= 1

ALPHABET = [chr(i) for i in range(ord('A'), ord('Z')+1)]  # ['A','B',...,'Z']
NUM_CLASSES = 26

print(f"Number of Training data : {len(train_data):,}")
print(f"Number of Testing data  : {len(test_data):,}")
print(f"Number of classes         : {NUM_CLASSES} ({ALPHABET[0]}–{ALPHABET[-1]})")
print(f"Image size        : {train_data[0][0].shape}")

Number of Training data : 124,800
Number of Testing data  : 20,800
Number of classes         : 26 (A–Z)
Image size        : torch.Size([1, 28, 28])


## 🔍 4. Dataset Exploration — Sample Visualization

In [4]:
# Show 1 sample per letter (A–Z)
sns.set_theme(style='dark', palette='muted')

fig, axes = plt.subplots(4, 7, figsize=(14, 8))
fig.suptitle('Dataset Samples EMNIST Letters (A–Z)', fontsize=16, fontweight='bold', y=1.01)

# Collect one image per class
shown = {}
for img, lbl in train_data:
    lbl = lbl.item() if isinstance(lbl, torch.Tensor) else lbl
    if lbl not in shown:
        shown[lbl] = img
    if len(shown) == NUM_CLASSES:
        break

for idx, ax in enumerate(axes.flat):
    if idx < NUM_CLASSES:
        img = shown[idx].squeeze().numpy()
        ax.imshow(img, cmap='inferno')
        ax.set_title(ALPHABET[idx], fontsize=13, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('result_v1/emnist_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Sample images saved to 'result_v1/emnist_samples.png'")

✅ Sample images saved to 'result_v1/emnist_samples.png'


In [5]:
# Classes distribution on training data
all_labels = train_data.targets.numpy()
unique, counts = np.unique(all_labels, return_counts=True)

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(
    [ALPHABET[i] for i in unique], counts,
    color=sns.color_palette('viridis', len(unique)),
    edgecolor='white', linewidth=0.6
)
ax.set_title('Class Distribution — EMNIST Letters Training Data', fontsize=14, fontweight='bold')
ax.set_xlabel('Letter', fontsize=12)
ax.set_ylabel('Number of Samples', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{cnt:,}', ha='center', va='bottom', fontsize=7.5)

plt.tight_layout()
plt.savefig('result_v1/_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔄 5. DataLoader

In [6]:
BATCH_SIZE_TRAIN = 128
BATCH_SIZE_TEST  = 512

# pin_memory=True accelerates CPU→GPU data transfer
train_loader = DataLoader(
    train_data, batch_size=BATCH_SIZE_TRAIN,
    shuffle=True, num_workers=4, pin_memory=True
)
test_loader  = DataLoader(
    test_data,  batch_size=BATCH_SIZE_TEST,
    shuffle=False, num_workers=4, pin_memory=True
)

print(f"Training batch  : {len(train_loader)} batch × {BATCH_SIZE_TRAIN} = {len(train_loader)*BATCH_SIZE_TRAIN:,} samples")
print(f"Testing batch   : {len(test_loader)} batch × {BATCH_SIZE_TEST} = {len(test_loader)*BATCH_SIZE_TEST:,} samples")

Training batch  : 975 batch × 128 = 124,800 samples
Testing batch   : 41 batch × 512 = 20,992 samples


## 🏗️ 6. ANN Architecture — MultilayerPerceptron

In [7]:
class MultilayerPerceptron(nn.Module):
    """
    4-layer ANN for classification EMNIST Letters (26 classes)
    
    Architecture:
        Input  : 784  (28×28 flattened)
        FC1    : 512  + BatchNorm + ReLU + Dropout(0.3)
        FC2    : 256  + BatchNorm + ReLU + Dropout(0.3)
        FC3    : 128  + BatchNorm + ReLU + Dropout(0.2)
        Output : 26   (A–Z)
    """
    def __init__(self, in_sz=784, out_sz=26, layers=[512, 256, 128]):
        super().__init__()
        
        self.fc1 = nn.Linear(in_sz,      layers[0])
        self.bn1 = nn.BatchNorm1d(layers[0])
        self.do1 = nn.Dropout(0.3)

        self.fc2 = nn.Linear(layers[0],  layers[1])
        self.bn2 = nn.BatchNorm1d(layers[1])
        self.do2 = nn.Dropout(0.3)

        self.fc3 = nn.Linear(layers[1],  layers[2])
        self.bn3 = nn.BatchNorm1d(layers[2])
        self.do3 = nn.Dropout(0.2)

        self.out = nn.Linear(layers[2],  out_sz)

        # Weight initialization with He (Kaiming) — suitable for ReLU
        for layer in [self.fc1, self.fc2, self.fc3, self.out]:
            nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')
            nn.init.zeros_(layer.bias)

    def forward(self, X):
        X = self.do1(F.relu(self.bn1(self.fc1(X))))
        X = self.do2(F.relu(self.bn2(self.fc2(X))))
        X = self.do3(F.relu(self.bn3(self.fc3(X))))
        return self.out(X)   # logits (without softmax — using CrossEntropyLoss)


torch.manual_seed(42)
model = MultilayerPerceptron().to(device)

# Architecture summary
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model is on       : {next(model.parameters()).device}")

MultilayerPerceptron(
  (fc1): Linear(in_features=784, out_features=512, bias=True)
  (bn1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (do1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (do2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=256, out_features=128, bias=True)
  (bn3): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (do3): Dropout(p=0.2, inplace=False)
  (out): Linear(in_features=128, out_features=26, bias=True)
)

Total parameters    : 571,290
Trainable parameters: 571,290
Model is on       : cuda:0


## ⚙️ 7. Loss Function, Optimizer and Scheduler

In [8]:
criterion = nn.CrossEntropyLoss()           # = LogSoftmax + NLLLoss
optimizer = torch.optim.Adam(
    model.parameters(), lr=1e-3, weight_decay=1e-4
)
# ReduceLROnPlateau: decrease LR if val_loss does not improve for 3 epochs
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

print("✅ Loss     : CrossEntropyLoss")
print("✅ Optimizer: Adam  (lr=1e-3, weight_decay=1e-4)")
print("✅ Scheduler: ReduceLROnPlateau (factor=0.5, patience=3)")

✅ Loss     : CrossEntropyLoss
✅ Optimizer: Adam  (lr=1e-3, weight_decay=1e-4)
✅ Scheduler: ReduceLROnPlateau (factor=0.5, patience=3)


## 🚀 8. Training Loop — with Detailed Progress per Batch & Epoch

In [9]:
EPOCHS      = 20
PRINT_EVERY = 100   # print per N batches

# Training history
history = {
    'train_loss': [], 'val_loss': [],
    'train_acc':  [], 'val_acc':  [],
    'lr':         []
}

best_val_loss  = float('inf')
CHECKPOINT_PATH = 'result_v1/best_emnist_model.pth'

# Helper: evaluate one epoch on any dataloader
def evaluate(loader):
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y  = X.to(device), y.to(device)
            logits = model(X.view(X.size(0), -1))
            loss   = criterion(logits, y)
            preds  = logits.argmax(dim=1)
            total_loss    += loss.item() * X.size(0)
            total_correct += (preds == y).sum().item()
            total_samples += X.size(0)
    return total_loss / total_samples, total_correct / total_samples * 100


# Main Training Loop
print(f"{'='*70}")
print(f"  Training ANN — EMNIST Letters  |  Device: {device}  |  Epochs: {EPOCHS}")
print(f"{'='*70}\n")

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, running_correct, running_samples = 0.0, 0, 0
    epoch_start = time.time()

    for b_idx, (X_train, y_train) in enumerate(train_loader, 1):
        X_train = X_train.to(device, non_blocking=True)
        y_train = y_train.to(device, non_blocking=True)

        # Forward pass
        logits = model(X_train.view(X_train.size(0), -1))
        loss   = criterion(logits, y_train)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        # Gradient clipping for stability
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # Accumulate statistics
        preds = logits.argmax(dim=1)
        running_correct += (preds == y_train).sum().item()
        running_samples += X_train.size(0)
        running_loss    += loss.item() * X_train.size(0)

        # ── Print every PRINT_EVERY batches ──
        if b_idx % PRINT_EVERY == 0:
            batch_acc  = running_correct / running_samples * 100
            batch_loss = running_loss    / running_samples
            total_imgs = running_samples
            print(
                f"  Epoch [{epoch:02d}/{EPOCHS}] "
                f"Batch [{b_idx:4d}/{len(train_loader)}] "
                f"[{total_imgs:6,}/{len(train_data):,}]  "
                f"Loss: {batch_loss:.6f}  "
                f"Acc: {batch_acc:6.2f}%"
            )

    # End of epoch evaluation
    train_loss = running_loss    / running_samples
    train_acc  = running_correct / running_samples * 100
    val_loss, val_acc = evaluate(test_loader)
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_loss)
    epoch_time = time.time() - epoch_start

    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        best_tag = " ⭐ BEST"
    else:
        best_tag = ""

    print(
        f"\n{'─'*70}\n"
        f"  ✅ Epoch {epoch:02d}/{EPOCHS} completed ({epoch_time:.1f}s)\n"
        f"     Train  — Loss: {train_loss:.6f}  Acc: {train_acc:.2f}%\n"
        f"     Val    — Loss: {val_loss:.6f}  Acc: {val_acc:.2f}%"
        f"{best_tag}\n"
        f"     Active LR: {current_lr:.2e}\n"
        f"{'─'*70}\n"
    )

total_time = time.time() - start_time
print(f"\n{'='*70}")
print(f"  🏁 Training completed dalam {total_time:.1f} seconds ({total_time/60:.1f} minutes)")
print(f"  💾 Best model saved to '{CHECKPOINT_PATH}'")
print(f"{'='*70}")

  Training ANN — EMNIST Letters  |  Device: cuda  |  Epochs: 20

  Epoch [01/20] Batch [ 100/975] [12,800/124,800]  Loss: 1.935014  Acc:  45.61%
  Epoch [01/20] Batch [ 200/975] [25,600/124,800]  Loss: 1.557159  Acc:  55.41%
  Epoch [01/20] Batch [ 300/975] [38,400/124,800]  Loss: 1.363166  Acc:  60.49%
  Epoch [01/20] Batch [ 400/975] [51,200/124,800]  Loss: 1.237066  Acc:  63.76%
  Epoch [01/20] Batch [ 500/975] [64,000/124,800]  Loss: 1.147011  Acc:  66.18%
  Epoch [01/20] Batch [ 600/975] [76,800/124,800]  Loss: 1.076693  Acc:  68.05%
  Epoch [01/20] Batch [ 700/975] [89,600/124,800]  Loss: 1.022013  Acc:  69.50%
  Epoch [01/20] Batch [ 800/975] [102,400/124,800]  Loss: 0.977258  Acc:  70.72%
  Epoch [01/20] Batch [ 900/975] [115,200/124,800]  Loss: 0.940001  Acc:  71.75%

──────────────────────────────────────────────────────────────────────
  ✅ Epoch 01/20 completed (13.4s)
     Train  — Loss: 0.914586  Acc: 72.45%
     Val    — Loss: 0.411477  Acc: 86.84% ⭐ BEST
     Active LR: 

## 📈 9. Training Visualization — Loss and Accuracy Curve (Advanced)

In [10]:
epochs_range = range(1, EPOCHS + 1)

fig = plt.figure(figsize=(16, 10))
gs  = GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

fig.suptitle(
    'Training History — ANN EMNIST Letters',
    fontsize=16, fontweight='bold', y=1.01
)

# Panel 1: Loss Curve 
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(epochs_range, history['train_loss'], 'o-', color='#4C72B0', lw=2, ms=5, label='Train Loss')
ax1.plot(epochs_range, history['val_loss'],   's--', color='#DD8452', lw=2, ms=5, label='Val Loss')
ax1.fill_between(epochs_range, history['train_loss'], history['val_loss'],
                 alpha=0.12, color='purple')
best_ep = np.argmin(history['val_loss']) + 1
ax1.axvline(best_ep, color='red', ls=':', lw=1.5, label=f'Best Epoch={best_ep}')
ax1.set_title('Loss per Epoch', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Panel 2: Accuracy Curve
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(epochs_range, history['train_acc'], 'o-', color='#4C72B0', lw=2, ms=5, label='Train Acc')
ax2.plot(epochs_range, history['val_acc'],   's--', color='#DD8452', lw=2, ms=5, label='Val Acc')
ax2.fill_between(epochs_range, history['train_acc'], history['val_acc'],
                 alpha=0.12, color='green')
ax2.axvline(best_ep, color='red', ls=':', lw=1.5, label=f'Best Epoch={best_ep}')
ax2.set_title('Accuracy per Epoch (%)', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_ylim([50, 100])
ax2.legend(); ax2.grid(True, alpha=0.3)

# Panel 3: Learning Rate Schedule
ax3 = fig.add_subplot(gs[1, 0])
ax3.semilogy(epochs_range, history['lr'], 'D-', color='#55A868', lw=2, ms=5)
ax3.set_title('Learning Rate Schedule (log scale)', fontweight='bold')
ax3.set_xlabel('Epoch'); ax3.set_ylabel('LR (log)')
ax3.grid(True, alpha=0.3, which='both')

# Panel 4: Gap Overfitting
ax4 = fig.add_subplot(gs[1, 1])
gap = [tr - va for tr, va in zip(history['train_acc'], history['val_acc'])]
colors_gap = ['#e74c3c' if g > 5 else '#2ecc71' for g in gap]
ax4.bar(epochs_range, gap, color=colors_gap, edgecolor='white', linewidth=0.5)
ax4.axhline(0, color='gray', lw=1)
ax4.axhline(5, color='red',  lw=1, ls='--', label='Overfitting threshold (5%)')
ax4.set_title('Gap Train–Val Accuracy (Overfitting Indicator)', fontweight='bold')
ax4.set_xlabel('Epoch'); ax4.set_ylabel('Gap (%)')
ax4.legend(); ax4.grid(True, alpha=0.3)

plt.savefig('result_v1/training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Training history graph saved to 'result_v1/training_history.png'")

✅ Training history graph saved to 'result_v1/training_history.png'


## 🧪 10. Evaluation on Test Data — Full Results

In [11]:
# Load best model, then run inference on entire test set
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()

all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device, non_blocking=True)
        logits = model(X.view(X.size(0), -1))
        probs  = F.softmax(logits, dim=1)
        preds  = probs.argmax(dim=1)
        all_preds.append(preds.cpu())
        all_labels.append(y)
        all_probs.append(probs.cpu())

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()
all_probs  = torch.cat(all_probs).numpy()

# Global metrics
test_acc = accuracy_score(all_labels, all_preds) * 100
prec, rec, f1, _ = precision_recall_fscore_support(
    all_labels, all_preds, average='macro'
)

print(f"{'='*50}")
print(f"  📊 EVALUATION RESULTS — TEST SET")
print(f"{'='*50}")
print(f"  Test Accuracy   : {test_acc:.3f}%")
print(f"  Macro Precision : {prec*100:.3f}%")
print(f"  Macro Recall    : {rec*100:.3f}%")
print(f"  Macro F1-Score  : {f1*100:.3f}%")
print(f"{'='*50}")

  📊 EVALUATION RESULTS — TEST SET
  Test Accuracy   : 92.587%
  Macro Precision : 92.667%
  Macro Recall    : 92.587%
  Macro F1-Score  : 92.591%


In [12]:
# Classification Report per classes
report = classification_report(
    all_labels, all_preds,
    target_names=ALPHABET
)
print("\n📄 Classification Report per Letter:")
print(report)


📄 Classification Report per Letter:
              precision    recall  f1-score   support

           A       0.92      0.93      0.93       800
           B       0.96      0.96      0.96       800
           C       0.96      0.95      0.96       800
           D       0.94      0.93      0.94       800
           E       0.97      0.95      0.96       800
           F       0.98      0.94      0.96       800
           G       0.88      0.77      0.82       800
           H       0.92      0.95      0.94       800
           I       0.76      0.73      0.74       800
           J       0.94      0.93      0.94       800
           K       0.95      0.94      0.95       800
           L       0.73      0.79      0.76       800
           M       0.97      0.97      0.97       800
           N       0.93      0.94      0.94       800
           O       0.94      0.97      0.95       800
           P       0.97      0.97      0.97       800
           Q       0.81      0.89      0.85 

In [ ]:
# Per-class accuracy as bar chart
_, _, f1_per_class, _ = precision_recall_fscore_support(
    all_labels, all_preds, average=None
)
prec_per, rec_per, _, _ = precision_recall_fscore_support(
    all_labels, all_preds, average=None
)

per_class_acc = []
for cls in range(NUM_CLASSES):
    mask = (all_labels == cls)
    acc  = (all_preds[mask] == cls).mean() * 100
    per_class_acc.append(acc)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Per-Class Evaluation — EMNIST Letters', fontsize=15, fontweight='bold')

# Left panel: Per-class Accuracy
palette = sns.diverging_palette(10, 130, n=NUM_CLASSES, as_cmap=False)
sorted_idx = np.argsort(per_class_acc)
axes[0].barh(
    [ALPHABET[i] for i in sorted_idx],
    [per_class_acc[i] for i in sorted_idx],
    color=[palette[j] for j in range(NUM_CLASSES)],
    edgecolor='white'
)
axes[0].axvline(test_acc, color='red', ls='--', lw=1.5, label=f'Average ({test_acc:.1f}%)')
axes[0].set_title('Accuracy per Letter (%)', fontweight='bold')
axes[0].set_xlabel('Accuracy (%)')
axes[0].legend()
axes[0].set_xlim([0, 105])

# Right panel: Precision vs Recall scatter
scatter = axes[1].scatter(
    prec_per * 100, rec_per * 100,
    c=f1_per_class * 100, cmap='RdYlGn',
    s=120, edgecolors='gray', linewidths=0.5, zorder=3
)
for i, letter in enumerate(ALPHABET):
    axes[1].annotate(letter, (prec_per[i]*100, rec_per[i]*100),
                     fontsize=8, ha='center', va='bottom', xytext=(0, 5),
                     textcoords='offset points')
cbar = plt.colorbar(scatter, ax=axes[1])
cbar.set_label('F1-Score (%)')
axes[1].set_title('Precision vs Recall per Letter', fontweight='bold')
axes[1].set_xlabel('Precision (%)')
axes[1].set_ylabel('Recall (%)')
axes[1].plot([0,100],[0,100], 'k--', alpha=0.2, label='Precision=Recall')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('result_v1/per_class_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔲 11. Confusion Matrix — Seaborn Heatmap

In [14]:
cm = confusion_matrix(all_labels, all_preds)
# Normalize per row (recall-normalized)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
fig.suptitle('Confusion Matrix — EMNIST Letters', fontsize=16, fontweight='bold')

# Raw counts
sns.heatmap(
    cm, ax=axes[0],
    xticklabels=ALPHABET, yticklabels=ALPHABET,
    annot=True, fmt='d', cmap='Blues',
    linewidths=0.4, linecolor='white',
    cbar_kws={'label': 'Number of Predictions'}
)
axes[0].set_title('Confusion Matrix (Raw Count)', fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# Normalized
sns.heatmap(
    cm_norm, ax=axes[1],
    xticklabels=ALPHABET, yticklabels=ALPHABET,
    annot=True, fmt='.2f', cmap='YlOrRd',
    linewidths=0.4, linecolor='white',
    vmin=0, vmax=1,
    cbar_kws={'label': 'Proportion (per row = per true class)'}
)
axes[1].set_title('Confusion Matrix (Normalized per True Class)', fontweight='bold')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.tight_layout()
plt.savefig('result_v1/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Confusion matrix saved to 'result_v1/confusion_matrix.png'")

✅ Confusion matrix saved to 'result_v1/confusion_matrix.png'


## 🔍 12. Visual Inspection — Correct and Incorrect Predictions

In [15]:
# Show 20 samples: 10 correct (green) + 10 incorrect (red)
all_images = test_data.data.numpy()   # (N, 28, 28) uint8

correct_mask   = (all_preds == all_labels)
incorrect_mask = ~correct_mask

correct_idx   = np.where(correct_mask)[0][:10]
incorrect_idx = np.where(incorrect_mask)[0][:10]

fig, axes = plt.subplots(2, 10, figsize=(20, 5))
fig.suptitle(
    'Test Set Prediction Inspection  |  🟢 Correct (top row)  |  🔴 Incorrect (bottom row)',
    fontsize=13, fontweight='bold'
)

for col, idx in enumerate(correct_idx):
    ax = axes[0, col]
    ax.imshow(all_images[idx], cmap='gray')
    conf = all_probs[idx, all_preds[idx]] * 100
    ax.set_title(f"{ALPHABET[all_preds[idx]]}\n{conf:.0f}%",
                 color='#27ae60', fontsize=9, fontweight='bold')
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_edgecolor('#27ae60'); spine.set_linewidth(2)

for col, idx in enumerate(incorrect_idx):
    ax = axes[1, col]
    ax.imshow(all_images[idx], cmap='gray')
    conf_pred = all_probs[idx, all_preds[idx]] * 100
    ax.set_title(
        f"True:{ALPHABET[all_labels[idx]]}\nPred:{ALPHABET[all_preds[idx]]}({conf_pred:.0f}%)",
        color='#e74c3c', fontsize=8, fontweight='bold'
    )
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_edgecolor('#e74c3c'); spine.set_linewidth(2)

plt.tight_layout()
plt.savefig('result_v1/prediction_inspection.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\Jeremia\AppData\Local\Temp\ipykernel_25412\3743775307.py:40: UserWarning: Glyph 128994 (\N{LARGE GREEN CIRCLE}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\Jeremia\AppData\Local\Temp\ipykernel_25412\3743775307.py:40: UserWarning: Glyph 128308 (\N{LARGE RED CIRCLE}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\Jeremia\AppData\Local\Temp\ipykernel_25412\3743775307.py:41: UserWarning: Glyph 128994 (\N{LARGE GREEN CIRCLE}) missing from font(s) Arial.
  plt.savefig('result_v1/prediction_inspection.png', dpi=150, bbox_inches='tight')
C:\Users\Jeremia\AppData\Local\Temp\ipykernel_25412\3743775307.py:41: UserWarning: Glyph 128308 (\N{LARGE RED CIRCLE}) missing from font(s) Arial.
  plt.savefig('result_v1/prediction_inspection.png', dpi=150, bbox_inches='tight')
C:\Users\Jeremia\AppData\Local\Temp\ipykernel_25412\3743775307.py:42: UserWarning: Glyph 128994 (\N{LARGE GREEN CIRCLE}) missing from font(s) Arial.
  plt.show()
C:\Users\Jeremia\AppData\Local\Temp\ip

## 📊 13. Final Summary Dashboard

In [16]:
# 1-page dashboard: all important metrics
fig = plt.figure(figsize=(20, 12))
gs  = GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.patch.set_facecolor('#1e1e2e')

DARK  = '#1e1e2e'
PANEL = '#2a2a3e'
BLUE  = '#4FC3F7'
ORG   = '#FFB74D'
GRN   = '#81C784'
RED   = '#EF9A9A'
WHITE = '#EEEEEE'

def styled_ax(ax):
    ax.set_facecolor(PANEL)
    for sp in ax.spines.values(): sp.set_color('#444')
    ax.tick_params(colors=WHITE, labelsize=8)
    ax.xaxis.label.set_color(WHITE); ax.yaxis.label.set_color(WHITE)
    ax.title.set_color(WHITE)
    ax.grid(True, alpha=0.15)
    return ax

# Loss Curve
ax1 = styled_ax(fig.add_subplot(gs[0, 0]))
ax1.plot(epochs_range, history['train_loss'], color=BLUE, lw=2, label='Train')
ax1.plot(epochs_range, history['val_loss'],   color=ORG,  lw=2, ls='--', label='Val')
ax1.set_title('Loss Curve', fontweight='bold')
ax1.legend(labelcolor=WHITE, facecolor=PANEL)

# Accuracy Curve
ax2 = styled_ax(fig.add_subplot(gs[0, 1]))
ax2.plot(epochs_range, history['train_acc'], color=GRN, lw=2, label='Train')
ax2.plot(epochs_range, history['val_acc'],   color=ORG, lw=2, ls='--', label='Val')
ax2.set_title('Accuracy Curve (%)', fontweight='bold')
ax2.legend(labelcolor=WHITE, facecolor=PANEL)

# LR Schedule
ax3 = styled_ax(fig.add_subplot(gs[0, 2]))
ax3.semilogy(epochs_range, history['lr'], color='#CE93D8', lw=2, marker='o', ms=4)
ax3.set_title('Learning Rate (log)', fontweight='bold')

# Per-class Accuracy (bar horizontal)
ax4 = styled_ax(fig.add_subplot(gs[1, :2]))
colors_bar = [GRN if a >= 80 else ORG if a >= 60 else RED for a in per_class_acc]
ax4.bar(ALPHABET, per_class_acc, color=colors_bar, edgecolor='none')
ax4.axhline(test_acc, color='white', ls=':', lw=1.5, label=f'Avg {test_acc:.1f}%')
ax4.set_title('Per-Class Accuracy (%)', fontweight='bold')
ax4.legend(labelcolor=WHITE, facecolor=PANEL)
ax4.set_ylim([0, 110])

# Mini Confusion Matrix
ax5 = fig.add_subplot(gs[1, 2])
ax5.set_facecolor(PANEL)
sns.heatmap(cm_norm, ax=ax5, cmap='YlOrRd', cbar=False,
            xticklabels=ALPHABET, yticklabels=ALPHABET,
            linewidths=0.2, linecolor='#333')
ax5.set_title('Confusion Matrix (Norm)', color=WHITE, fontweight='bold')
ax5.tick_params(colors=WHITE, labelsize=6.5)

# Summary Stats Text Box
ax6 = fig.add_subplot(gs[2, :])
ax6.set_facecolor(PANEL)
ax6.axis('off')
summary_text = (
    f"📊  FINAL RESULT SUMMARY\n"
    f"────────────────────────────────────────────────────────────\n"
    f"  Dataset         : EMNIST Letters (A–Z, 26 classes)\n"
    f"  Model           : Multilayer Perceptron (ANN)  |  "
    f"  Params          : {trainable_params:,}\n"
    f"  Device          : {str(device).upper()}\n"
    f"  Epochs          : {EPOCHS}\n"
    f"────────────────────────────────────────────────────────────\n"
    f"  Test Accuracy   : {test_acc:.3f}%   |   "
    f"  Best Val Loss   : {best_val_loss:.6f}\n"
    f"  Macro Precision : {prec*100:.2f}%       |   "
    f"  Macro Recall    : {rec*100:.2f}%\n"
    f"  Macro F1-Score  : {f1*100:.2f}%\n"
    f"  Best Letter     : {ALPHABET[np.argmax(per_class_acc)]} ({max(per_class_acc):.1f}%)"
    f"   |   Worst Letter : {ALPHABET[np.argmin(per_class_acc)]} ({min(per_class_acc):.1f}%)"
)
ax6.text(0.02, 0.5, summary_text, transform=ax6.transAxes,
         fontsize=11, color=WHITE, va='center',
         fontfamily='monospace',
         bbox=dict(facecolor='#12121f', boxstyle='round,pad=0.5', alpha=0.7))

fig.suptitle(
    '🧠 ANN EMNIST Letters — Final Project Dashboard',
    fontsize=17, fontweight='bold', color=WHITE, y=1.01
)

plt.savefig('result_v1/final_dashboard.png', dpi=150, bbox_inches='tight',
            facecolor=DARK)
plt.show()
print("✅ Final dashboard saved to 'result_v1/final_dashboard.png'")

C:\Users\Jeremia\AppData\Local\Temp\ipykernel_25412\359313677.py:93: UserWarning: Glyph 128202 (\N{BAR CHART}) missing from font(s) DejaVu Sans Mono.
  plt.savefig('result_v1/final_dashboard.png', dpi=150, bbox_inches='tight',
C:\Users\Jeremia\AppData\Local\Temp\ipykernel_25412\359313677.py:93: UserWarning: Glyph 129504 (\N{BRAIN}) missing from font(s) Arial.
  plt.savefig('result_v1/final_dashboard.png', dpi=150, bbox_inches='tight',
C:\Users\Jeremia\AppData\Local\Temp\ipykernel_25412\359313677.py:95: UserWarning: Glyph 128202 (\N{BAR CHART}) missing from font(s) DejaVu Sans Mono.
  plt.show()
C:\Users\Jeremia\AppData\Local\Temp\ipykernel_25412\359313677.py:95: UserWarning: Glyph 129504 (\N{BRAIN}) missing from font(s) Arial.
  plt.show()


✅ Final dashboard saved to 'result_v1/final_dashboard.png'


## 💾 14. Save Model and Export Results to CSV

In [17]:
# Save complete model
torch.save({
    'epoch':      EPOCHS,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'history':    history,
    'test_acc':   test_acc,
}, 'result_v1/emnist_ann_final.pth')

# Export training history to CSV
df_hist = pd.DataFrame({
    'epoch':      list(epochs_range),
    'train_loss': history['train_loss'],
    'val_loss':   history['val_loss'],
    'train_acc':  history['train_acc'],
    'val_acc':    history['val_acc'],
    'lr':         history['lr'],
})
df_hist.to_csv('result_v1/training_history.csv', index=False)

# Export per-class report to CSV
df_report = pd.DataFrame({
    'letter':    ALPHABET,
    'accuracy':  per_class_acc,
    'precision': prec_per * 100,
    'recall':    rec_per  * 100,
    'f1_score':  f1_per_class * 100,
}).sort_values('f1_score', ascending=False)
df_report.to_csv('result_v1/per_class_report.csv', index=False)

print("✅ Files saved:")
print("   - emnist_ann_final.pth     (complete checkpoint)")
print("   - training_history.csv     (training history)")
print("   - per_class_report.csv     (laporan per letter)")
print("\n📋 Top 5 Letter (F1-Score):")
print(df_report.head(5).to_string(index=False))
print("\n📋 Bottom 5 Letter (F1-Score):")
print(df_report.tail(5).to_string(index=False))

✅ Files saved:
   - emnist_ann_final.pth     (complete checkpoint)
   - training_history.csv     (training history)
   - per_class_report.csv     (laporan per letter)

📋 Top 5 Letter (F1-Score):
letter  accuracy  precision  recall  f1_score
     Z    98.125  97.153465  98.125 97.636816
     S    97.125  97.735849  97.125 97.429467
     P    96.750  97.358491  96.750 97.053292
     M    96.875  97.117794  96.875 96.996245
     W    96.375  96.981132  96.375 96.677116

📋 Bottom 5 Letter (F1-Score):
letter  accuracy  precision  recall  f1_score
     A    92.750  92.288557  92.750 92.518703
     Q    88.750  80.681818  88.750 84.523810
     G    76.875  88.362069  76.875 82.219251
     L    78.500  72.769409  78.500 75.526158
     I    72.625  75.552666  72.625 74.059911
